# TASK-021 — Kaggle / Qwen3.8:27B / source nhúng sẵn

**Import một file này rồi Run All. Không cần GitHub, EXPECTED_COMMIT hoặc ZIP/Dataset source.**

Trước khi chạy: bật Internet và chọn GPU (ưu tiên T4 x2 nếu được cấp).
Mặc định chạy unit + HTTP mock + regression + live text/ảnh với `qwen3.8:27b`.
Nếu muốn test C++ trước, đặt `RUN_LIVE=False`; phần đó không cần GPU hoặc model.
Ngrok là tùy chọn khi cần gọi Ollama từ máy cá nhân, mặc định tắt.

**NOT RUN — pending Kaggle validation.** Không có kết quả thực thi sẵn trong file.
Notebook chứa snapshot source hiện tại; sau khi sửa source cần tạo lại notebook.

Tham khảo yêu cầu của bạn: [tdquang / qwen-3-8-27b-ngrok-ollama](https://www.kaggle.com/code/tdquang/qwen-3-8-27b-ngrok-ollama).
Trang này chưa truy xuất được bằng công cụ lúc soạn, nên đây là notebook mới theo luồng
Ollama/Qwen, không phải bản sao đã kiểm chứng từng cell của notebook tham khảo.
Đối chiếu [Ollama Linux](https://docs.ollama.com/linux), [FAQ/GPU/ngrok](https://docs.ollama.com/faq),
[đúng model qwen3.8:27b](https://ollama.com/library/qwen3.8:27b).


In [ ]:
RUN_LIVE = True
LIVE_MODEL = "qwen3.8:27b"
CONTEXT_LENGTH = 16384
INSTALL_TOOLCHAIN = True
TOOLCHAIN = "/kaggle/working/task021-toolchain"
RUN_ROOT_REGRESSION = True
ENABLE_NGROK = False
OLLAMA_URL = "http://127.0.0.1:11434"


## 1. Khởi tạo và lưu evidence
Chỉ thực thi trên Kaggle. Mỗi run dùng thư mục mới, giữ log và trạng thái riêng.


In [ ]:
import base64
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
import uuid
import xml.etree.ElementTree as ET
from zipfile import ZipFile, ZIP_DEFLATED

if not Path("/kaggle/working").is_dir() or not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    raise RuntimeError("Chỉ thực thi trong Kaggle Notebook; không chạy local/sandbox.")
WORK = Path("/kaggle/working") / ("task021-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime()) + "-" + uuid.uuid4().hex[:8])
EVIDENCE = WORK / "evidence"
EVIDENCE.mkdir(parents=True)
REPO = WORK / "AI-Agent"
ZIP = WORK.with_suffix(".zip")
report = {"source_mode": "embedded", "requested_model": LIVE_MODEL, "commands": [],
          "statuses": {name: "NOT RUN" for name in
                       ("toolchain", "source", "build", "task17_unit", "task21_unit",
                        "task17_http_integration", "task17_live_criteria", "root_regression", "gpu", "ollama", "live", "ngrok")}}


def archive():
    # Chỉ đóng gói evidence được chọn; không chứa repo, token, model hoặc biến môi trường.
    required = ("toolchain", "source", "build", "task17_unit", "task21_unit",
                "task17_http_integration", "task17_live_criteria", "root_regression")
    core_states = [report["statuses"][name] for name in required]
    report["core_status"] = ("FAIL" if "FAIL" in core_states else "BLOCKED" if "BLOCKED" in core_states
                             else "PASS" if all(state == "PASS" for state in core_states) else "NOT RUN")
    if RUN_LIVE:
        required += ("gpu", "ollama", "live")
    if ENABLE_NGROK:
        required += ("ngrok",)
    states = [report["statuses"][name] for name in required]
    report["overall_status"] = ("FAIL" if "FAIL" in states else "BLOCKED" if "BLOCKED" in states
                                else "PASS" if all(state == "PASS" for state in states) else "NOT RUN")
    (EVIDENCE / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    with ZipFile(ZIP, "w", ZIP_DEFLATED) as output:
        for path in sorted(EVIDENCE.rglob("*")):
            if path.is_file():
                output.write(path, path.relative_to(EVIDENCE))


def run_logged(name, command, cwd=None, check=True, timeout=1200, env=None):
    # Ghi cả stdout/stderr và exit code; thất bại vẫn xuất ZIP trước khi dừng.
    args = [str(part) for part in command]
    print("[RUN]", name, flush=True)
    try:
        result = subprocess.run(args, cwd=cwd, text=True, capture_output=True, timeout=timeout, env=env)
    except Exception as error:
        for stream, attribute in (("stdout", "stdout"), ("stderr", "stderr")):
            partial = getattr(error, attribute, "") or ""
            if isinstance(partial, bytes):
                partial = partial.decode("utf-8", errors="replace")
            (EVIDENCE / (name + "." + stream + ".txt")).write_text(partial, encoding="utf-8")
        report["commands"].append({"name": name, "status": "FAIL", "error": str(error)})
        archive()
        raise
    (EVIDENCE / (name + ".stdout.txt")).write_text(result.stdout, encoding="utf-8")
    (EVIDENCE / (name + ".stderr.txt")).write_text(result.stderr, encoding="utf-8")
    report["commands"].append({"name": name, "returncode": result.returncode,
                               "status": "PASS" if result.returncode == 0 else "FAIL"})
    archive()
    print("[END]", name, "exit=", result.returncode, flush=True)
    if result.returncode:
        print(result.stderr[-2000:] or result.stdout[-2000:])
    if check and result.returncode:
        raise RuntimeError(f"{name}: exit {result.returncode}. Xem log trong {ZIP}")
    return result


def copy_test_artifacts(build, prefix):
    # Chép JSON unit/HTTP và CTest XML/log kể cả khi có test FAIL.
    destination = EVIDENCE / prefix
    destination.mkdir(exist_ok=True)
    for path in build.glob("*.json"):
        shutil.copy2(path, destination / path.name)
    cache_file = build / "CMakeCache.txt"
    if cache_file.is_file():
        shutil.copy2(cache_file, destination / cache_file.name)
    testing = build / "Testing"
    if testing.exists():
        for path in testing.rglob("*"):
            if path.is_file() and path.suffix in (".xml", ".log", ".txt"):
                target = destination / path.relative_to(build)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)


def ctest_statuses(build):
    # Đọc kết quả CTest thực tế; test thiếu hoặc skipped không được xem là PASS.
    statuses = {}
    for xml in (build / "Testing").glob("*/Test.xml"):
        for test in ET.parse(xml).findall("./Testing/Test"):
            statuses[test.findtext("Name")] = "PASS" if test.get("Status") == "passed" else "FAIL"
    return statuses


archive()
print("Evidence:", ZIP)


def api_json(endpoint, payload=None, timeout=30):
    # Chỉ dùng cho setup/metadata; extraction thực tế vẫn qua OllamaClient C++.
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(OLLAMA_URL + endpoint, data=body,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)


## 2. Khôi phục source nhúng sẵn
Kiểm hash ZIP và từng file trước khi build. Base commit chỉ để tham chiếu vì source chưa commit.


In [ ]:
def copy_uploaded_source(selected):
    # Chỉ chép file có trong manifest, kiểm đường dẫn và hash; không extractall ZIP.
    zipped = selected.is_file()
    package = ZipFile(selected) if zipped else None
    try:
        raw = (package.read("task021-source-manifest.json") if zipped
               else (selected / "task021-source-manifest.json").read_bytes())
        snapshot = json.loads(raw)
        if snapshot.get("schema_version") != 1 or snapshot.get("kind") != "task021-working-tree":
            raise ValueError("Manifest không phải gói source TASK-021 được hỗ trợ")
        files = snapshot.get("files")
        if not isinstance(files, dict) or not files:
            raise ValueError("Manifest thiếu danh sách file")
        identity = hashlib.sha256(json.dumps(files, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
        if identity != snapshot.get("source_id"):
            raise ValueError("source_id không khớp manifest")
        if zipped:
            names = [item.filename for item in package.infolist() if not item.is_dir()]
            expected = {"task021-source-manifest.json"} | {"AI-Agent/" + name for name in files}
            if len(names) != len(set(names)) or set(names) != expected:
                raise ValueError("ZIP có file trùng, thừa hoặc thiếu so với manifest")
        REPO.mkdir()
        for relative, expected_hash in files.items():
            name = PurePosixPath(relative)
            if (not relative or name.is_absolute() or name.as_posix() != relative or
                    any(part in ("..", ".git") for part in name.parts) or
                    "\\" in relative or ":" in relative):
                raise ValueError("Path source không hợp lệ: " + relative)
            if not isinstance(expected_hash, str) or not re.fullmatch(r"[0-9a-f]{64}", expected_hash):
                raise ValueError("Hash source không hợp lệ: " + relative)
            if zipped:
                content = package.read("AI-Agent/" + relative)
            else:
                source_file = selected / "AI-Agent" / relative
                if not source_file.resolve().is_relative_to((selected / "AI-Agent").resolve()):
                    raise ValueError("Source nằm ngoài thư mục upload")
                content = source_file.read_bytes()
            if hashlib.sha256(content).hexdigest() != expected_hash:
                raise ValueError("Hash source không khớp: " + relative)
            target = REPO / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
        (EVIDENCE / "uploaded_source_manifest.json").write_bytes(raw)
        report["source_path"] = str(selected)
        report["source_id"] = identity
        report["base_commit"] = snapshot.get("base_commit")
        report["commit"] = None  # Đây là working tree, không phải source của base_commit.
        if zipped:
            report["source_zip_sha256"] = hashlib.sha256(selected.read_bytes()).hexdigest()
        return files
    finally:
        if package is not None:
            package.close()


EMBEDDED_SHA256 = '4a3cc7b981c3f529f8a154b65b28b14cd419062b8de8b1a6cb5f796c3d57fe21'
SOURCE_BASE64 = (
    '...'
)

try:
    source_bytes = base64.b64decode(SOURCE_BASE64, validate=True)
    if hashlib.sha256(source_bytes).hexdigest() != EMBEDDED_SHA256:
        raise ValueError("Payload source trong notebook bị thay đổi hoặc bị cắt; import lại file gốc.")
    source_zip = WORK / "embedded-source.zip"
    source_zip.write_bytes(source_bytes)
    manifest = copy_uploaded_source(source_zip)
    (EVIDENCE / "source_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    for name in ("src/agent/deadline_confirmation.cpp", "include/agent/deadline_confirmation.h",
                 "tests/deadline_extractor/deadline_confirmation_test.cpp"):
        if name not in manifest:
            raise ValueError("Snapshot thiếu " + name)
    if "add_test(NAME task21_unit " not in (REPO / "tests/deadline_extractor/CMakeLists.txt").read_text(encoding="utf-8"):
        raise ValueError("Snapshot chưa đăng ký task21_unit")
    report["statuses"]["source"] = "PASS"
    print("Source ready:", len(manifest), "files; source_id:", report["source_id"])
except Exception as error:
    report["statuses"]["source"] = "BLOCKED"
    report["error"] = str(error)
    print("Source BLOCKED:", error)
finally:
    archive()


## 3. Toolchain C++23
Cài compiler GCC >=15, libcurl và SQLite trên Kaggle. Probe cả `std::expected` và `<print>` trước build.


In [ ]:
try:
    toolchain = Path(TOOLCHAIN).resolve()
    # Tên executable phải là micromamba để shell hook và run kích hoạt đúng môi trường.
    mamba = Path("/kaggle/working/task021-mamba/bin/micromamba")
    mamba.parent.mkdir(parents=True, exist_ok=True)
    os.environ["MAMBA_EXE"] = str(mamba)
    os.environ["MAMBA_ROOT_PREFIX"] = "/kaggle/working/task021-mamba-root"
    os.environ["PATH"] = str(mamba.parent) + os.pathsep + os.environ.get("PATH", "")
    if INSTALL_TOOLCHAIN:
        if not mamba.exists():
            package = WORK / "micromamba.tar.bz2"
            urllib.request.urlretrieve("https://micro.mamba.pm/api/micromamba/linux-64/latest", package)
            report["micromamba_package_sha256"] = hashlib.sha256(package.read_bytes()).hexdigest()
            with tarfile.open(package, "r:bz2") as source:
                member = source.extractfile("bin/micromamba")
                if member is None:
                    raise RuntimeError("Thiếu bin/micromamba trong package")
                mamba.write_bytes(member.read())
            mamba.chmod(0o755)
        if not (toolchain / "conda-meta").is_dir():
            run_logged("toolchain_install", [mamba, "create", "-y", "-p", toolchain, "-c", "conda-forge",
                       "gxx_linux-64>=15", "cmake", "pkg-config", "libcurl", "sqlite", "libsqlite"], timeout=1800)
    CMAKE_OPTIONS = ["-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_EXPORT_COMPILE_COMMANDS=ON"]
    if mamba.exists() and (toolchain / "conda-meta").is_dir():
        compilers = sorted((toolchain / "bin").glob("*conda-linux-gnu-c++"))
        if not compilers:
            raise RuntimeError("Toolchain không có compiler C++")
        CXX = str(compilers[0])
        CMAKE_BIN = str(toolchain / "bin/cmake")
        CTEST_BIN = str(toolchain / "bin/ctest")
        if not Path(CMAKE_BIN).is_file() or not Path(CTEST_BIN).is_file():
            raise RuntimeError("Toolchain thiếu cmake/ctest; cần cài đủ package trên Kaggle")
        PREFIX = [str(mamba), "run", "-p", str(toolchain)]
        # Kiểm activation thực sự, không chỉ tin exit code khi shell hook báo lỗi.
        activation = run_logged("toolchain_activation", PREFIX + [sys.executable, "-c",
            "import os,json; print(json.dumps({'prefix':os.environ.get('CONDA_PREFIX')}))"])
        if Path(json.loads(activation.stdout)["prefix"]).resolve() != toolchain:
            raise RuntimeError("micromamba chưa kích hoạt đúng CONDA_PREFIX")
        curl_library = toolchain / "lib/libcurl.so"
        sqlite_library = toolchain / "lib/libsqlite3.so"
        for dependency in (toolchain / "include/curl/curl.h", toolchain / "include/sqlite3.h",
                           curl_library, sqlite_library):
            if not dependency.is_file():
                raise RuntimeError("Toolchain thiếu dependency: " + str(dependency))
        # Giữ headers/libraries cùng prefix với compiler, tránh kéo /usr/include vào sysroot Conda.
        CMAKE_OPTIONS += ["-DCMAKE_PREFIX_PATH=" + str(toolchain),
                          "-DCMAKE_BUILD_RPATH=" + str(toolchain / "lib"),
                          "-DCURL_NO_CURL_CMAKE=ON",
                          "-DCURL_INCLUDE_DIR=" + str(toolchain / "include"),
                          "-DCURL_LIBRARY=" + str(curl_library),
                          "-DCURL_LIBRARY_RELEASE=" + str(curl_library),
                          "-DSQLite3_INCLUDE_DIR=" + str(toolchain / "include"),
                          "-DSQLite3_LIBRARY=" + str(sqlite_library)]
    else:
        CXX = os.environ.get("CXX") or shutil.which("g++")
        CMAKE_BIN = shutil.which("cmake")
        CTEST_BIN = shutil.which("ctest")
        PREFIX = []
    if not CXX or not CMAKE_BIN or not CTEST_BIN:
        raise RuntimeError("Thiếu compiler/cmake/ctest; cài toolchain trên Kaggle")
    CMAKE_OPTIONS += ["-DCMAKE_CXX_COMPILER=" + CXX]
    report["cmake_options"] = CMAKE_OPTIONS
    report["compiler"] = run_logged("compiler_version", PREFIX + [CXX, "--version"]).stdout
    report["cmake"] = run_logged("cmake_version", PREFIX + [CMAKE_BIN, "--version"]).stdout
    probe = WORK / "expected_probe.cpp"
    probe.write_text("#include <expected>\nint main(){std::expected<int,int> x=1; return x.value()-1;}\n", encoding="utf-8")
    run_logged("expected_probe_build", PREFIX + [CXX, "-std=c++23", str(probe), "-o", str(WORK / "expected_probe")])
    run_logged("expected_probe_run", PREFIX + [WORK / "expected_probe"])
    print_probe = WORK / "print_probe.cpp"
    print_probe.write_text("#include <print>\nint main(){std::print(\"{}\", 1); return 0;}\n", encoding="utf-8")
    run_logged("print_probe_build", PREFIX + [CXX, "-std=c++23", str(print_probe), "-lstdc++exp", "-o", str(WORK / "print_probe")])
    run_logged("print_probe_run", PREFIX + [WORK / "print_probe"])
    # Preflight dùng đúng CMake và các dependency của dự án, có chrono/time_t/curl/SQLite.
    dependency_source = WORK / "dependency-probe"
    dependency_source.mkdir()
    (dependency_source / "main.cpp").write_text(
        "#include <chrono>\n#include <expected>\n#include <print>\n#include <curl/curl.h>\n#include <sqlite3.h>\n"
        "int main(){std::expected<int,int> x=1; std::print(\"{}\", x.value()); "
        "auto t=std::chrono::system_clock::to_time_t(std::chrono::system_clock::now()); "
        "(void)t; if(curl_global_init(CURL_GLOBAL_DEFAULT)!=0) return 1; "
        "curl_global_cleanup(); return sqlite3_initialize();}\n", encoding="utf-8")
    (dependency_source / "CMakeLists.txt").write_text(
        "cmake_minimum_required(VERSION 3.20)\nproject(dependency_probe LANGUAGES CXX)\n"
        "set(CMAKE_CXX_STANDARD 23)\nset(CMAKE_CXX_STANDARD_REQUIRED ON)\n"
        "find_package(CURL REQUIRED)\nfind_package(SQLite3 REQUIRED)\n"
        "add_executable(dependency_probe main.cpp)\n"
        "target_link_libraries(dependency_probe PRIVATE CURL::libcurl SQLite::SQLite3)\n"
        "if(CMAKE_CXX_COMPILER_ID STREQUAL \"GNU\")\n"
        "target_link_libraries(dependency_probe PRIVATE stdc++exp)\nendif()\n", encoding="utf-8")
    dependency_build = WORK / "dependency-probe-build"
    run_logged("dependency_probe_configure", PREFIX + [CMAKE_BIN, "-S", dependency_source, "-B", dependency_build] + CMAKE_OPTIONS)
    # Lưu các cache quan trọng; không ghi toàn bộ biến môi trường hoặc secret.
    cache = {}
    for line in (dependency_build / "CMakeCache.txt").read_text(encoding="utf-8").splitlines():
        if not line.startswith(("#", "//")) and ":" in line and "=" in line:
            key = line.split(":", 1)[0]
            if key in ("CURL_INCLUDE_DIR", "CURL_LIBRARY", "SQLite3_INCLUDE_DIR", "SQLite3_LIBRARY"):
                cache[key] = line.split("=", 1)[1]
    if PREFIX:
        for key in ("CURL_INCLUDE_DIR", "CURL_LIBRARY", "SQLite3_INCLUDE_DIR", "SQLite3_LIBRARY"):
            if key not in cache or not Path(cache[key]).resolve().is_relative_to(toolchain):
                raise RuntimeError("Dependency bị lấy ngoài toolchain: " + key)
    report["resolved_dependencies"] = cache
    (EVIDENCE / "resolved_dependencies.json").write_text(json.dumps(cache, indent=2), encoding="utf-8")
    run_logged("dependency_probe_build", PREFIX + [CMAKE_BIN, "--build", dependency_build, "--parallel", "2", "--verbose"])
    run_logged("dependency_probe_run", PREFIX + [dependency_build / "dependency_probe"])
    report["statuses"]["toolchain"] = "PASS"
except Exception as error:
    report["statuses"]["toolchain"] = "BLOCKED"
    report["error"] = str(error)
    print("Toolchain BLOCKED:", error)
finally:
    archive()


## 4. Build và test TASK-017/TASK-021
Gồm `task17_unit`, `task21_unit`, HTTP mock và bộ kiểm tiêu chí bằng fixture. Fixture test không phải live model.


In [ ]:
BUILD = REPO / "build/task21"
if report["statuses"]["source"] != "PASS" or report["statuses"]["toolchain"] != "PASS":
    report["statuses"]["build"] = "BLOCKED"
    print("Build BLOCKED: cần source và toolchain PASS.")
    archive()
else:
    BUILD = REPO / "build/task21"
    try:
        run_logged("configure_task21", PREFIX + [CMAKE_BIN, "-S", "tests/deadline_extractor", "-B", "build/task21",
                   ] + CMAKE_OPTIONS, cwd=REPO)
        run_logged("build_task21", PREFIX + [CMAKE_BIN, "--build", "build/task21", "--parallel", "2"], cwd=REPO)
        report["statuses"]["build"] = "PASS"
        result = run_logged("ctest_task21", PREFIX + [CTEST_BIN, "--test-dir", "build/task21", "--output-on-failure",
                            "--no-tests=error", "-T", "Test"], cwd=REPO, check=False)
        statuses = ctest_statuses(BUILD)
        for name in ("task17_unit", "task21_unit", "task17_http_integration", "task17_live_criteria"):
            report["statuses"][name] = statuses.get(name, "NOT RUN")
        if result.returncode or any(report["statuses"][name] != "PASS" for name in
                                   ("task17_unit", "task21_unit", "task17_http_integration", "task17_live_criteria")):
            raise RuntimeError("CTest thất bại hoặc thiếu kết quả; xem XML/JSON/log")
    except Exception as error:
        if report["statuses"]["build"] != "PASS":
            report["statuses"]["build"] = "FAIL"
        report["error"] = str(error)
        print("TASK-021 build/test FAIL:", error)
    finally:
        copy_test_artifacts(BUILD, "task21")
        archive()


## 5. Regression toàn dự án
Dùng CMake gốc; lỗi được báo riêng và không tự sửa source của đồng đội.


In [ ]:
if RUN_ROOT_REGRESSION and report["statuses"]["source"] == "PASS" and report["statuses"]["toolchain"] == "PASS":
    ROOT_BUILD = REPO / "build/regression"
    try:
        run_logged("configure_root", PREFIX + [CMAKE_BIN, "-S", ".", "-B", "build/regression",
                   "-DBUILD_TESTING=ON"] + CMAKE_OPTIONS, cwd=REPO)
        run_logged("build_root", PREFIX + [CMAKE_BIN, "--build", "build/regression", "--parallel", "2"], cwd=REPO)
        result = run_logged("ctest_root", PREFIX + [CTEST_BIN, "--test-dir", "build/regression",
                            "--output-on-failure", "--no-tests=error", "-T", "Test"], cwd=REPO, check=False)
        root_statuses = ctest_statuses(ROOT_BUILD)
        report["root_tests"] = root_statuses
        required = ("agent_contract", "scheduler", "trajectory_ics", "memory_tool", "ollama_client", "ollama_http")
        if result.returncode or not root_statuses or any(value != "PASS" for value in root_statuses.values()) or any(name not in root_statuses for name in required):
            raise RuntimeError("Regression root có test thiếu/FAIL/skipped; không sửa shared source để né lỗi")
        report["statuses"]["root_regression"] = "PASS"
    except Exception as error:
        report["statuses"]["root_regression"] = "FAIL"
        report["root_error"] = str(error)
        print("Root regression FAIL:", error)
    finally:
        copy_test_artifacts(ROOT_BUILD, "regression")
        archive()


## 6. GPU → Ollama → tải và warm-up model

Model `qwen3.8:27b` tải khoảng 18 GB theo thư viện Ollama; thời gian và khả năng chạy
phụ thuộc tài nguyên thực tế được Kaggle cấp. Notebook ghi VRAM từng GPU, digest,
capabilities và bộ nhớ model dùng; không tự đổi model để vượt lỗi.
Nếu server được tái sử dụng thì cấu hình môi trường cũ của server vẫn giữ nguyên.
Đợi cell tải model hoàn tất; log được giữ trong evidence.


In [ ]:
if not RUN_LIVE:
    print("Bỏ qua cài Ollama/model vì RUN_LIVE=False; live giữ NOT RUN.")
else:
    try:
        gpu = run_logged("gpu_inventory", ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free",
                                          "--format=csv,noheader,nounits"], check=False)
        if gpu.returncode or not gpu.stdout.strip():
            raise RuntimeError("Chưa có GPU NVIDIA: chọn GPU trong Kaggle Settings rồi chạy lại.")
        report["gpu_inventory"] = gpu.stdout.strip().splitlines()
        report["statuses"]["gpu"] = "PASS"
        print(gpu.stdout)
        if not shutil.which("ollama"):
            run_logged("apt_update", ["apt-get", "update"], timeout=600)
            run_logged("ollama_dependencies", ["apt-get", "install", "-y", "curl", "ca-certificates", "zstd", "pciutils"], timeout=600)
            installer = WORK / "install_ollama.sh"
            urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
            report["ollama_installer_sha256"] = hashlib.sha256(installer.read_bytes()).hexdigest()
            run_logged("ollama_install", ["sh", installer], timeout=1200)
        ollama_env = os.environ.copy()
        ollama_env.update({"OLLAMA_HOST": "127.0.0.1:11434",
                           "OLLAMA_MODELS": "/kaggle/working/task021-ollama-models",
                           "OLLAMA_CONTEXT_LENGTH": str(CONTEXT_LENGTH),
                           "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_FLASH_ATTENTION": "1",
                           "OLLAMA_KV_CACHE_TYPE": "q8_0", "OLLAMA_NO_CLOUD": "1"})
        try:
            version = api_json("/api/version", timeout=5)
            report["ollama_server"] = "reused existing server; its original environment is retained"
        except Exception:
            server_log = open(EVIDENCE / "ollama_server.log", "a", encoding="utf-8")
            ollama_process = subprocess.Popen(["ollama", "serve"], env=ollama_env,
                                              stdout=server_log, stderr=subprocess.STDOUT)
            report["ollama_server"] = "started by this notebook"
            for attempt in range(120):
                if ollama_process.poll() is not None:
                    raise RuntimeError("Ollama dừng khi khởi động; xem ollama_server.log.")
                try:
                    version = api_json("/api/version", timeout=2)
                    break
                except Exception:
                    time.sleep(1)
            else:
                raise RuntimeError("Ollama chưa sẵn sàng sau 120 giây.")
        report["ollama_version"] = version
        run_logged("ollama_cli_version", ["ollama", "--version"], env=ollama_env)
        tags = api_json("/api/tags", timeout=30)
        selected = next((item for item in tags.get("models", [])
                         if item.get("name") == LIVE_MODEL or item.get("model") == LIVE_MODEL), None)
        if selected is None:
            run_logged("ollama_pull", ["ollama", "pull", LIVE_MODEL], env=ollama_env, timeout=3600)
        info = api_json("/api/show", {"model": LIVE_MODEL}, timeout=30)
        (EVIDENCE / "model_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
        if not {"completion", "vision", "tools"}.issubset(set(info.get("capabilities", []))):
            raise RuntimeError("Model chưa báo đủ completion/vision/tools; không tự đổi model.")
        tags = api_json("/api/tags", timeout=30)
        selected = next((item for item in tags.get("models", [])
                         if item.get("name") == LIVE_MODEL or item.get("model") == LIVE_MODEL), None)
        if not selected or not selected.get("digest"):
            raise RuntimeError("Không xác định được digest của đúng model yêu cầu.")
        report["model"] = LIVE_MODEL
        report["model_digest"] = selected["digest"]
        report["model_size_bytes"] = selected.get("size")
        report["model_capabilities"] = info.get("capabilities", [])
        print("Warm-up model:", LIVE_MODEL, flush=True)
        warmup = api_json("/api/chat", {"model": LIVE_MODEL, "stream": False, "think": False,
                            "messages": [{"role": "user", "content": "Chỉ trả lời OK."}],
                            "options": {"num_ctx": CONTEXT_LENGTH, "num_predict": 16},
                            "keep_alive": "30m"}, timeout=900)
        (EVIDENCE / "warmup.json").write_text(json.dumps(warmup, ensure_ascii=False, indent=2), encoding="utf-8")
        if not warmup.get("message", {}).get("content", "").strip():
            raise RuntimeError("Warm-up chưa trả nội dung; xem warmup.json.")
        process_info = api_json("/api/ps", timeout=30)
        (EVIDENCE / "ollama_ps.json").write_text(json.dumps(process_info, indent=2), encoding="utf-8")
        loaded = next((item for item in process_info.get("models", [])
                       if item.get("name") == LIVE_MODEL or item.get("model") == LIVE_MODEL), None)
        if not loaded or loaded.get("size_vram", 0) <= 0:
            raise RuntimeError("Chưa xác minh model dùng GPU qua /api/ps; xem ollama_ps.json.")
        report["model_vram_bytes"] = loaded.get("size_vram")
        report["statuses"]["ollama"] = "PASS"
        print("Ollama ready:", LIVE_MODEL, "VRAM bytes:", loaded.get("size_vram"))
    except Exception as error:
        if report["statuses"]["gpu"] != "PASS":
            report["statuses"]["gpu"] = "BLOCKED"
        report["statuses"]["ollama"] = "BLOCKED"
        report["ollama_error"] = str(error)
        print("Ollama BLOCKED:", error)
    finally:
        archive()


## 7. Live text và ảnh qua Client C++
Probe hiện có gọi OllamaClient; giữ JSON và raw response, không sửa đầu ra model để chấm PASS.
Unit TASK-021 kiểm luồng xác nhận deterministic; live này kiểm extraction text/ảnh TASK-017.


In [ ]:
if RUN_LIVE:
    try:
        if report["statuses"]["ollama"] != "PASS" or report["statuses"]["build"] != "PASS":
            raise RuntimeError("Live cần Ollama và build PASS; xem nhóm bị lỗi ở trên.")
        if not LIVE_MODEL:
            raise ValueError("Cần LIVE_MODEL tường minh")
        run_logged("gpu_live", ["nvidia-smi"])
        # Chỉ đọc metadata; extraction vẫn đi qua OllamaClient C++ hiện có.
        with urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=30) as response:
            report["ollama_version"] = json.load(response)
        show_request = urllib.request.Request(OLLAMA_URL + "/api/show",
            data=json.dumps({"model": LIVE_MODEL}).encode(), headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(show_request, timeout=30) as response:
            metadata = json.load(response)
        report["model"] = LIVE_MODEL
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=30) as response:
            tags = json.load(response)
        selected = next((model for model in tags.get("models", [])
                         if model.get("name") in (LIVE_MODEL, LIVE_MODEL + ":latest")
                         or model.get("model") in (LIVE_MODEL, LIVE_MODEL + ":latest")), None)
        if not selected or not selected.get("digest"):
            raise RuntimeError("Chưa xác định được digest của model đang dùng")
        report["model_digest"] = selected["digest"]
        report["model_modified_at"] = selected.get("modified_at")
        report["model_capabilities"] = metadata.get("capabilities", [])
        report["model_details"] = metadata.get("details", {})
        if not {"completion", "vision", "tools"}.issubset(set(metadata.get("capabilities", []))):
            raise RuntimeError("Model chưa xác nhận đủ completion/vision/tools")
        import importlib.util
        spec = importlib.util.spec_from_file_location("task17_checks", REPO / "tests/deadline_extractor/live_result_checks.py")
        checks = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(checks)
        fixtures = REPO / "tests/fixtures/deadline_extractor"
        expected = json.loads((fixtures / "live_expected.json").read_text(encoding="utf-8"))["items"]
        repository_expected = json.loads((fixtures / "input_expected.json").read_text(encoding="utf-8"))["items"]
        vision_expected = json.loads((REPO / "assets/task013/timetable_expected.json").read_text(encoding="utf-8"))
        live_cases = [
            ("text_complete", "--text", fixtures / "live_syllabus.txt", None),
            ("text_missing_year", "--text", REPO / "assets/input.txt", None),
            ("text_context_2026", "--text", REPO / "assets/input.txt", "2026"),
            ("vision_timetable", "--image", REPO / "assets/timetable.png", None)]
        live_results = []
        for name, mode, source, year in live_cases:
            result = run_logged(name, PREFIX + [BUILD / "deadline_extractor_probe", mode, source, OLLAMA_URL,
                                LIVE_MODEL] + ([year] if year else []), check=False, timeout=1200)
            try:
                if result.returncode:
                    raise RuntimeError("Probe exit " + str(result.returncode))
                data = json.loads(result.stdout)
                (EVIDENCE / (name + ".json")).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
                (EVIDENCE / (name + ".raw.txt")).write_text(data.get("raw_model_content", ""), encoding="utf-8")
                if name == "text_complete":
                    checks.check_complete_text(data, source.read_text(encoding="utf-8"), expected)
                elif mode == "--text":
                    checks.check_repository_text(data, source.read_text(encoding="utf-8"), repository_expected,
                                                 reference_year=int(year) if year else None)
                else:
                    checks.check_vision_timetable(data, vision_expected)
                live_results.append({"name": name, "status": "PASS"})
            except Exception as error:
                live_results.append({"name": name, "status": "FAIL", "reason": str(error)})
        report["live_cases"] = live_results
        report["statuses"]["live"] = "PASS" if all(case["status"] == "PASS" for case in live_results) else "FAIL"
    except Exception as error:
        report["statuses"]["live"] = "BLOCKED"
        report["live_error"] = str(error)
    finally:
        archive()
else:
    print("Live: NOT RUN; không suy diễn PASS từ unit/HTTP mock.")


## 8. Ngrok tùy chọn
Chỉ cần khi gọi server từ máy cá nhân. Nếu bật `ENABLE_NGROK=True`, thêm Kaggle Secret tên `NGROK_AUTHTOKEN` và cấp quyền cho notebook. Không dán token vào cell.


In [ ]:
if ENABLE_NGROK:
    try:
        if report["statuses"]["ollama"] != "PASS":
            raise RuntimeError("Ollama chưa sẵn sàng để mở tunnel.")
        run_logged("pyngrok_install", [sys.executable, "-m", "pip", "install", "-q", "pyngrok"], timeout=600)
        from kaggle_secrets import UserSecretsClient
        from pyngrok import ngrok
        ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_AUTHTOKEN"))
        tunnel = ngrok.connect(addr="127.0.0.1:11434", proto="http", bind_tls=True,
                               host_header="localhost:11434")
        print("Endpoint cho máy cá nhân:", tunnel.public_url)
        report["statuses"]["ngrok"] = "PASS"
    except Exception as error:
        report["statuses"]["ngrok"] = "BLOCKED"
        print("Ngrok chưa sẵn sàng; kiểm Kaggle Secret NGROK_AUTHTOKEN và log.")
    finally:
        archive()
else:
    print("Ngrok tắt; probe C++ gọi http://127.0.0.1:11434 ngay trong Kaggle.")


## 9. Kết quả và tải ZIP
`Core TASK-021` gồm build/unit/HTTP/fixture/root regression. `Overall` còn yêu cầu live khi RUN_LIVE=True; model chưa chạy không được suy thành PASS.


In [ ]:
archive()
rows = [{"Nhóm": name, "Trạng thái": state} for name, state in report["statuses"].items()]
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    print(json.dumps(rows, ensure_ascii=False, indent=2))
print("Core TASK-021:", report["core_status"])
print("Overall:", report["overall_status"])
for label in ("error", "root_error", "ollama_error", "live_error"):
    if report.get(label):
        print(label + ":", report[label])
for relative in ("task21_unit_results.json", "unit_results.json", "http_results.json", "criteria_results.json"):
    artifact = BUILD / relative
    if artifact.is_file():
        data = json.loads(artifact.read_text(encoding="utf-8"))
        print(relative, "PASS:", data.get("passed"), "FAIL:", data.get("failed"))
print("Evidence ZIP:", ZIP)
from IPython.display import FileLink, display
display(FileLink(str(ZIP)))
print("Tải ZIP trong /kaggle/working về để review. ZIP không chứa model weights hoặc secret.")
